In [ ]:
import subprocess, sys, os, glob, shutil, hashlib, time
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print(r.stdout[-1500:], r.stderr[-800:], flush=True); return r
sh("find /kaggle/input -maxdepth 4 | head -40")

In [ ]:
W = "/kaggle/working/w"; shutil.rmtree(W, ignore_errors=True); os.makedirs(W)
py = glob.glob("/kaggle/input/**/cider_thuong_8_11.py", recursive=True)
if py:
    S = os.path.dirname(os.path.dirname(py[0]))
    for d in ("harness", "runs"): shutil.copytree(f"{S}/{d}", f"{W}/{d}")
else:   # thư mục còn ở dạng zip
    for z in glob.glob("/kaggle/input/**/cider-thuong-8-11/*.zip", recursive=True) + glob.glob("/kaggle/input/**/cider_thuong*/*.zip", recursive=True):
        sh(f"unzip -o -q {z} -d {W}/{os.path.basename(z)[:-4]}")
sh(f"find {W} -maxdepth 2 | head -30")
assert os.path.exists(f"{W}/harness/cider_thuong_8_11.py"), "thiếu mã"
TR = [p for p in glob.glob("/kaggle/input/**/p1_train_rows.jsonl", recursive=True)]
print(TR); TR = TR[0]
print("md5", hashlib.md5(open(TR, "rb").read()).hexdigest(), "(bản máy nhà 6898b21e767e1c4f1d165d0bf6c71875)")

In [ ]:
r = sh("java -version 2>&1 | grep -c 'version \"1.8' ")
if r.stdout.strip() != "1":
    sh("apt-get update -qq && apt-get install -y -qq openjdk-8-jre-headless 2>&1 | tail -2")
jv = sorted(glob.glob("/usr/lib/jvm/*8*")); print(jv)
if jv: os.environ["JAVA_HOME"] = jv[0]; os.environ["PATH"] = jv[0] + "/bin:" + os.environ["PATH"]
sh("java -version")
sh(f"{sys.executable} -m pip install -q pycocoevalcap 2>&1 | tail -2")

In [ ]:
p = subprocess.Popen(f"cd {W}/harness && {sys.executable} -u cider_thuong_8_11.py --runs {W}/runs --train {TR} --out {W}/cider_thuong_8_11.json --B 2000 > {W}/log.txt 2>&1", shell=True)
while p.poll() is None:
    time.sleep(120); print(time.strftime("%H:%M"), open(f"{W}/log.txt").read()[-400:].replace("\n", " | "), flush=True)
print(open(f"{W}/log.txt").read()[-4000:]); print("exit", p.returncode)

In [ ]:
for f in ("cider_thuong_8_11.json", "log.txt"):
    if os.path.exists(f"{W}/{f}"): shutil.copyfile(f"{W}/{f}", f"/kaggle/working/{f}")
shutil.rmtree(W); print(os.listdir("/kaggle/working"))